# exp088 sequence model residual diversity train

Lightweight GRU/TCN residual correction audit on top of the exp073 deterministic OOF anchor.

## Contents

1. Setup and configuration
2. Input source check
3. Sequence residual audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json

import pandas as pd
import torch

from settings import ExperimentPaths, get_nested
from sequence_model_residual_diversity import candidate_paths, find_first_existing, run_audit

paths = ExperimentPaths()
config = paths.config
print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('status:', get_nested(config, 'experiment.status'))
print('parent:', get_nested(config, 'lineage.parent'))
print('cache_parent:', get_nested(config, 'lineage.cache_parent'))
print('selected mode/model:', get_nested(config, 'audit.selected_mode'), get_nested(config, 'audit.selected_model'))
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available(), 'default_dtype:', torch.get_default_dtype())
print('variants:', [v['name'] for v in get_nested(config, 'model.variants')])

## 2. Input source check

In [ ]:
prediction_path = find_first_existing(candidate_paths(config, 'exp073_oof_predictions'), 'exp073_oof_predictions')
feature_cache_path = find_first_existing(candidate_paths(config, 'exp072_feature_cache'), 'exp072_feature_cache')
print('prediction_path:', prediction_path)
print('feature_cache_path:', feature_cache_path)
print('prediction_size_mb:', round(prediction_path.stat().st_size / 1024 / 1024, 2))
print('feature_cache_size_mb:', round(feature_cache_path.stat().st_size / 1024 / 1024, 2))
print('prediction columns:')
display(pd.read_csv(prediction_path, nrows=0).head())
print('feature cache first columns:')
display(pd.read_csv(feature_cache_path, nrows=0).iloc[:, :20])

## 3. Sequence residual audit

In [ ]:
summary = run_audit()
print(json.dumps({
    'rows': summary['rows'],
    'wells': summary['wells'],
    'best_prediction': summary['best_prediction'],
    'best_rmse_tvt': summary['best_rmse_tvt'],
    'baseline_rmse_tvt': summary['baseline_rmse_tvt'],
    'elapsed_seconds': summary['elapsed_seconds'],
}, indent=2, sort_keys=True))

## 4. Metrics and artifacts

In [ ]:
artifact_dir = paths.artifacts_dir
prefix = get_nested(config, 'audit.output_prefix')
metrics = pd.read_csv(artifact_dir / f'{prefix}_metrics.csv')
bucket_metrics = pd.read_csv(artifact_dir / f'{prefix}_bucket_metrics.csv')
diversity = pd.read_csv(artifact_dir / f'{prefix}_diversity_metrics.csv')
display(metrics.sort_values('rmse_tvt'))
display(bucket_metrics.sort_values(['distance_bucket', 'rmse_tvt']).head(30))
display(diversity.sort_values('baseline_error_corr'))
print('artifacts:')
for path in sorted(artifact_dir.glob(f'{prefix}*')):
    print(path.name, round(path.stat().st_size / 1024 / 1024, 3), 'MB')